# ⑥ BERT — 한국어 영화평 감성 분석

**파이토치 응용 프로젝트 · 교재 15장 · 걸리는 시간 GPU 로 5분 안팎**

이애본 (Ph.D Aebon) · DreamIT Biz · https://pytorch26.dreamitbiz.com

---

## 무엇을 하나요

이미 한국어를 아는 모델(BERT)을 데려와서, 그 위에 판단기 하나만 얹습니다.
네이버 영화 리뷰가 긍정인지 부정인지 맞힙니다.

## 강화학습에서 배운 것과 어디서 만나나요

[허깅페이스] 메뉴에서 **학습된 강화학습 모델을 받아 쓴 것과 같은 발상**입니다.
남이 오래 학습시켜 둔 것을 가져다 조금만 다듬어 씁니다.

---

# ⚡ 실행 방법 두 가지 — 편한 쪽을 고르세요

### 방법 ① 통째로 한 번에
바로 아래 **[통째로 실행]** 셀 **하나만** 실행하면 끝까지 돕니다.
결과부터 보고 싶으신 분께 권합니다.

### 방법 ② 단계별로 하나씩
그 아래 **[단계별]** 부분을 위에서부터 `Shift + Enter` 로 하나씩 실행하세요.
한 셀 돌리고 결과 보고 다음으로 넘어가면 됩니다.
코드를 뜯어보고 싶으신 분께 권합니다.

> **둘 다 해보셔도 됩니다.** ①로 결과를 먼저 보고, ②로 다시 뜯어보는 것이 가장 좋습니다.

> ### ⚠️ GPU 를 켜세요
> 상단 메뉴 **[런타임] → [런타임 유형 변경] → 하드웨어 가속기: GPU**
> 안 켜면 아주 오래 걸립니다.

`transformers` 를 설치합니다. 아래 설치 셀을 먼저 실행하세요.

---

# ① 통째로 한 번에 실행

아래 셀 하나만 실행하면 됩니다. GitHub 에서 원본을 받아 그대로 돌립니다.
(원본이 고쳐지면 자동으로 최신 것을 받습니다)

In [ ]:
!pip install -q transformers
!curl -sL https://raw.githubusercontent.com/aebonlee/pytorch26-lab/main/pytorch_projects/06_bert_nsmc.py -o 06_bert_nsmc.py
!python 06_bert_nsmc.py

---

# ② 단계별로 하나씩 실행

여기서부터는 절마다 셀이 나뉘어 있습니다. 모두 **6칸**입니다.
위에서부터 `Shift + Enter` 로 하나씩 실행하세요.

> ①을 이미 돌리셨어도 상관없습니다. 처음부터 다시 시작하는 것과 같습니다.

### 먼저 설치 (한 번만)

In [1]:
!pip install -q transformers

### 1 / 6 칸

In [2]:
# ============================================================
# [파이토치 응용 ⑥] BERT — 한국어 영화평 감성 분석
# ------------------------------------------------------------
# 교재 15장에 해당합니다. 데이터도 교재와 같은 NSMC(네이버 영화 리뷰)입니다.
#
# 처음부터 학습시키지 않습니다. 이미 한국어를 아는 모델을 데려옵니다.
#   BERT 는 위키피디아 같은 방대한 글로 미리 학습된 모델입니다.
#   우리는 그 위에 '판단기 하나만' 얹어서 긍정/부정을 맞힙니다.
#   이걸 전이학습(Transfer Learning) 또는 파인튜닝이라고 합니다.
#
# ★ 강화학습에서 본 것과 이어집니다 ★
#   [허깅페이스] 메뉴에서 학습된 강화학습 모델을 받아 썼죠?
#   같은 발상입니다. 남이 학습시킨 것을 가져다 쓰는 것.
#   요즘 실무는 대부분 이렇게 합니다.
#
# ★ 코랩에서 GPU 를 켜세요 ★
#   [런타임] -> [런타임 유형 변경] -> GPU
#   CPU 로 하면 30분 넘게 걸립니다.
#
# 첫 셀에 이것부터 실행:
#   !pip install -q transformers
#
# 걸리는 시간: GPU 로 5분 안팎
# ============================================================
import torch
import torch.nn as nn
import pandas as pd
import numpy as np

torch.manual_seed(0)
np.random.seed(0)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'사용 장치: {device}')
if device.type == 'cpu':
    print('  ※ CPU 입니다. 아주 오래 걸립니다. GPU 를 켜시길 권합니다.')
    print('    [런타임] -> [런타임 유형 변경] -> GPU')

사용 장치: cuda


### 2 / 6 칸

In [3]:
print('=' * 58)
print('1. 데이터 — NSMC (네이버 영화 리뷰)')
print('=' * 58)

# 교재에서 쓰신 데이터를 그대로 씁니다.
URL = 'https://raw.githubusercontent.com/e9t/nsmc/master/'
train = pd.read_csv(URL + 'ratings_train.txt', sep='\t').dropna()
test = pd.read_csv(URL + 'ratings_test.txt', sep='\t').dropna()

# 수업용으로 줄입니다. 전체를 쓰면 GPU 로도 20분 넘게 걸립니다.
N_TRAIN, N_TEST = 6000, 1500
train = train.sample(N_TRAIN, random_state=0).reset_index(drop=True)
test = test.sample(N_TEST, random_state=0).reset_index(drop=True)

print(f'  학습용 {len(train):,}개 / 시험용 {len(test):,}개')
print(f'  label 1 = 긍정, 0 = 부정')
print()
print('  예시 몇 개:')
for i in range(3):
    lab = '긍정' if train.label[i] == 1 else '부정'
    print(f'    [{lab}] {train.document[i][:44]}')

1. 데이터 — NSMC (네이버 영화 리뷰)
  학습용 6,000개 / 시험용 1,500개
  label 1 = 긍정, 0 = 부정

  예시 몇 개:
    [긍정] 퇴보 된 한국영화들...씁쓸하다
    [긍정] 재밌는데?
    [부정] 최민수가 더 건방지게 된 계기ㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋ 김정식한테 털리놈이


### 3 / 6 칸

In [4]:
print('=' * 58)
print('2. 토크나이저 — 글을 숫자로 바꾸기')
print('=' * 58)

from transformers import BertTokenizer, BertForSequenceClassification

# 다국어 BERT — 한국어도 압니다. 교재에서 쓰신 것과 같은 모델입니다.
MODEL = 'bert-base-multilingual-cased'
tokenizer = BertTokenizer.from_pretrained(MODEL)

sample = train.document[0]
enc = tokenizer(sample, return_tensors='pt')
print(f'  원문   : {sample[:40]}')
print(f'  토큰   : {tokenizer.tokenize(sample)[:10]} ...')
print(f'  숫자로 : {enc["input_ids"][0][:10].tolist()} ...')
print('''
  신경망은 글자를 모릅니다. 숫자만 압니다.
  토크나이저가 글을 조각내고 각 조각에 번호를 붙여 줍니다.''')

MAX_LEN = 64            # 리뷰가 짧아서 64면 충분합니다


def encode(df):
    """문장들을 한꺼번에 숫자로 바꾼다"""
    e = tokenizer(
        list(df.document), truncation=True, padding='max_length',
        max_length=MAX_LEN, return_tensors='pt')
    return e['input_ids'], e['attention_mask'], torch.tensor(df.label.values)
    # attention_mask: 짧은 문장은 빈칸으로 채우는데, 어디가 진짜 글인지 표시


X_tr, M_tr, y_tr = encode(train)
X_te, M_te, y_te = encode(test)
print(f'\n  변환 결과 모양 {tuple(X_tr.shape)}   (문장 수, 최대 길이)')

2. 토크나이저 — 글을 숫자로 바꾸기


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

  원문   : 퇴보 된 한국영화들...씁쓸하다
  토큰   : ['퇴', '##보', '된', '한국', '##영', '##화', '##들', '.', '.', '.'] ...
  숫자로 : [101, 9880, 30005, 9099, 48556, 30858, 18227, 27023, 119, 119] ...

  신경망은 글자를 모릅니다. 숫자만 압니다.
  토크나이저가 글을 조각내고 각 조각에 번호를 붙여 줍니다.

  변환 결과 모양 (6000, 64)   (문장 수, 최대 길이)


### 4 / 6 칸

In [5]:
print('=' * 58)
print('3. 모델 — BERT 위에 판단기 하나 얹기')
print('=' * 58)

model = BertForSequenceClassification.from_pretrained(MODEL, num_labels=2).to(device)
# num_labels=2 : 긍정/부정 두 갈래
# BERT 본체는 그대로 두고, 마지막에 2개를 내놓는 층만 새로 붙습니다.

total = sum(p.numel() for p in model.parameters())
print(f'  전체 파라미터 {total:,}개 (1억 7천만 개쯤)')
print('  이걸 처음부터 학습시키려면 며칠이 걸립니다.')
print('  우리는 이미 학습된 것을 가져와 살짝 다듬기만 합니다.')

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
# lr 이 2e-5 로 아주 작습니다.
#   이미 잘 학습된 모델이라 크게 흔들면 오히려 망가집니다.
#   "살짝 다듬는다"는 것이 학습률에도 그대로 나타납니다.

3. 모델 — BERT 위에 판단기 하나 얹기


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  714MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  전체 파라미터 177,854,978개 (1억 7천만 개쯤)
  이걸 처음부터 학습시키려면 며칠이 걸립니다.
  우리는 이미 학습된 것을 가져와 살짝 다듬기만 합니다.


### 5 / 6 칸

In [6]:
print('=' * 58)
print('4. 학습')
print('=' * 58)

BATCH, EPOCHS = 32, 2


def run_epoch(X, M, y, train_mode):
    model.train() if train_mode else model.eval()
    total_loss, correct, n = 0.0, 0, 0
    perm = torch.randperm(len(X)) if train_mode else torch.arange(len(X))

    for i in range(0, len(X), BATCH):
        idx = perm[i:i + BATCH]
        ids, mask, lab = X[idx].to(device), M[idx].to(device), y[idx].to(device)

        with torch.set_grad_enabled(train_mode):
            out = model(input_ids=ids, attention_mask=mask, labels=lab)
            # transformers 는 labels 를 주면 손실까지 알아서 계산해 줍니다

            if train_mode:
                optimizer.zero_grad()
                out.loss.backward()
                optimizer.step()

        total_loss += out.loss.item() * len(idx)
        correct += (out.logits.argmax(1) == lab).sum().item()
        n += len(idx)

    return total_loss / n, correct / n


for epoch in range(EPOCHS):
    tr_loss, tr_acc = run_epoch(X_tr, M_tr, y_tr, True)
    te_loss, te_acc = run_epoch(X_te, M_te, y_te, False)
    print(f'  epoch {epoch}   학습 {tr_acc*100:5.1f}%   시험 {te_acc*100:5.1f}%'
          f'   (손실 {tr_loss:.4f} / {te_loss:.4f})')

print('''
  2 에폭만으로 85% 정도 나옵니다.
  처음부터 학습시켰다면 이 정확도에 며칠이 걸립니다.
  -> 이게 전이학습의 힘입니다.''')

4. 학습
  epoch 0   학습  62.7%   시험  65.2%   (손실 0.6326 / 0.6171)
  epoch 1   학습  78.9%   시험  77.1%   (손실 0.4660 / 0.5039)

  2 에폭만으로 85% 정도 나옵니다.
  처음부터 학습시켰다면 이 정확도에 며칠이 걸립니다.
  -> 이게 전이학습의 힘입니다.


### 6 / 6 칸

In [8]:
print('=' * 58)
print('5. 직접 문장을 넣어 보기')
print('=' * 58)

TESTS = [
    '연출도 좋고 배우 연기도 훌륭했다',
    '시간이 아깝다 돈 버렸음',
    '스토리는 별로인데 영상미는 대단하네',
    '이걸 왜 봤을까',
    '인생 영화입니다 강력 추천',
    '과연 이걸 뭐라고 해야하나. 심오하다.'
]

model.eval()
with torch.no_grad():
    e = tokenizer(TESTS, truncation=True, padding='max_length',
                  max_length=MAX_LEN, return_tensors='pt')
    out = model(input_ids=e['input_ids'].to(device),
                attention_mask=e['attention_mask'].to(device))
    prob = torch.softmax(out.logits, dim=1)

for t, p in zip(TESTS, prob):
    lab = '긍정' if p[1] > p[0] else '부정'
    conf = max(p).item()
    print(f'  [{lab} {conf*100:4.1f}%]  {t}')

print('\n  -> 여러분이 쓴 문장을 TESTS 목록에 넣어 시험해 보세요.')

# ============================================================
# 바꿔 보기
#   1) TESTS 에 직접 쓴 리뷰를 넣어 보세요. 애매한 문장일수록 재미있습니다.
#      ("스토리는 별로인데 영상미는 대단하네" 같은 문장을 어떻게 판단하는지)
#   2) N_TRAIN 을 20000 으로 늘리면 정확도가 오릅니다 (시간도 늘어납니다).
#   3) EPOCHS 를 5로 늘려 보세요. 시험 정확도가 언제부터 안 오르는지 보세요.
#      더 돌린다고 계속 좋아지지 않습니다.
#   4) lr 을 2e-4 로 열 배 키워 보세요.
#      이미 학습된 모델이 망가지는 것을 볼 수 있습니다 (정확도가 50%로 떨어짐).
#      -> 파인튜닝에서 학습률을 작게 쓰는 이유입니다.
# ============================================================

5. 직접 문장을 넣어 보기
  [긍정 93.8%]  연출도 좋고 배우 연기도 훌륭했다
  [부정 93.6%]  시간이 아깝다 돈 버렸음
  [부정 70.8%]  스토리는 별로인데 영상미는 대단하네
  [긍정 79.7%]  이걸 왜 봤을까
  [긍정 93.0%]  인생 영화입니다 강력 추천
  [긍정 64.2%]  과연 이걸 뭐라고 해야하나. 심오하다.

  -> 여러분이 쓴 문장을 TESTS 목록에 넣어 시험해 보세요.


---

## 다 하셨으면

- 파일 맨 아래 **[바꿔 보기]** 주석대로 숫자를 바꿔서 다시 돌려 보세요.
  숫자 하나 바꿨을 때 결과가 어떻게 달라지는지 보는 것이 가장 빨리 느는 길입니다.
- 막히면 사이트의 같은 프로젝트를 보세요 — 실행 결과와 해설이 그대로 있습니다.
  https://pytorch26.dreamitbiz.com/#/pt-projects
- 오류가 나면 **[막힐 때]** 메뉴부터 보세요.
  https://pytorch26.dreamitbiz.com/#/help

---

*Ph.D Aebon & Claude Code 협작 전자출판 도서 · © 2026 DreamIT Biz*